# TAPE Fluorescence

In [1]:
from protein_benchmark_models.data import OneHotSequenceDataset

import pandas as pd
from torch.utils.data import DataLoader

### Configure Dataset & Dataloader

In [2]:
# Train dataset
train_df = pd.read_csv(".data/fluorescence/train.csv")
sequences = train_df["sequence"].tolist()
targets = train_df["target"].to_numpy()
max_seq_len = train_df["sequence"].map(lambda x: len(x)).max()
train_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

# Valid dataset
val_df = pd.read_csv(".data/fluorescence/valid.csv")
sequences = val_df["sequence"].tolist()
targets = val_df["target"].to_numpy()
val_dataset = OneHotSequenceDataset(
    sequences=sequences,
    targets=targets,
    seq_len=max_seq_len
)

### Model Registry

In [3]:
from protein_benchmark_models.models import ModelRegistry
print(ModelRegistry.list())

/Users/dylanelliott/workspace/protein-benchmark-models/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['ridge_regressor', 'mlp_regressor']


### Ridge Regressor

In [4]:
# Create model
model = ModelRegistry.get("ridge_regressor")(
    alpha=1.0
)

# Train, evaluate, and save
model.train(
    experiment_name="tape-fluorescence",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=".models/tape_fluorescence_ridge_regressor",
    seed=1,
    tracking=True
)

[ridge_regressor] Train X: (21446, 5214)
[ridge_regressor] Train y: (21446,)
[ridge_regressor] Training model
[ridge_regressor] Evaluating model
[ridge_regressor] Valid X: (5362, 5214)
[ridge_regressor] Valid y: (5362,)
[ridge_regressor] Valid RMSE: 0.3744
[ridge_regressor] Valid R2: 0.7993
[ridge_regressor] Valid SpearmanR: 0.8203
🏃 View run gifted-bear-963 at: http://localhost:5000/#/experiments/2/runs/2cdb475b927f41cd80c1c5a0338b1f6c
🧪 View experiment at: http://localhost:5000/#/experiments/2


### MLP Regressor

In [ ]:
# Create model
model = ModelRegistry.get("mlp_regressor")(
    layer_dims = [max_seq_len * len(train_dataset.vocab), 1024, 1],
    hidden_activation = "ReLU",
    output_activation = "Identity",
    use_bias = True,
    norm = "layer"
)

# Train, evaluate, and save
model.train(
    experiment_name="tape-fluorescence",
    train_data=train_dataset,
    val_data=val_dataset,
    model_path=".models/tape_fluorescence_mlp_regressor",
    lr=1e-4,
    weight_decay = 0.01,
    max_epochs=1000,
    batch_size=256,
    val_frequency=1,
    patience=25,
    save_model="best",
    seed=42,
    tracking=True
)

Epoch: 100/100 | train_loss: 0.0335 | val_loss: 0.0993 | best_val_loss: 0.0872: 100%|██████████| 100/100 [14:41<00:00,  8.81s/it]


🏃 View run wistful-cat-301 at: http://localhost:5000/#/experiments/2/runs/910625588d2b48528c5b749f0d56c9c3
🧪 View experiment at: http://localhost:5000/#/experiments/2
